<a href="https://colab.research.google.com/github/youngbean0519/Self-Project/blob/main/notebooks/10_kobert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path("/content/Self-Project")
SRC_DIR = PROJECT_DIR / "src"

print("Python:", sys.version)

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/youngbean0519/Self-Project.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )

# tokenizers는 소스 컴파일 대신 빌드된 wheel 사용
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "--only-binary=tokenizers",
        "transformers==4.57.6",
        "tokenizers>=0.22.0,<=0.23.0",
        "sentencepiece",
        "kiwipiepy",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "git+https://github.com/SKTBrain/KoBERT.git"
        "#egg=kobert_tokenizer&subdirectory=kobert_hf",
    ],
    check=True,
)

for path in [PROJECT_DIR, SRC_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

print("설치 완료")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
설치 완료


In [ ]:
import copy
import gc
import hashlib
import json
import math
import random
import time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers

from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score,
)
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import BertModel, get_linear_schedule_with_warmup
from kobert_tokenizer import KoBERTTokenizer

import preprocessing.text as text_preprocessing
import utils.labels as utils_labels

DATA_DIR = Path("/content/drive/MyDrive/26_KOTE/data")
ARTIFACT_DIR = Path("/content/drive/MyDrive/26_KOTE/artifacts/week9_kobert")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_NAMES = list(utils_labels.LABELS)
NUM_LABELS = 44
MODEL_NAME = "skt/kobert-base-v1"

assert len(LABEL_NAMES) == NUM_LABELS
assert DEVICE.type == "cuda", "Colab 런타임을 GPU로 변경해 주세요."

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("결과 저장:", ARTIFACT_DIR)

def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

def save_json(data, path):
  with open(path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

def config_id(config):
  text = json.dumps(config, sort_keys=True)
  return hashlib.sha256(text.encode()).hexdigest()[:12]

def synchronize():
  torch.cuda.synchronize()

def autocast_context():
  if DEVICE.type == "cuda":
    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    )
  return nullcontext()

PyTorch: 2.11.0+cu130
Transformers: 4.57.6
GPU: NVIDIA A100-SXM4-80GB
결과 저장: /content/drive/MyDrive/26_KOTE/artifacts/week9_kobert


In [ ]:
def load_split(name):
    return pd.read_csv(
        DATA_DIR / f"{name}.tsv",
        sep="\t",
        header=None,
        names=["id", "text", "labels"],
    )


train_df = load_split("train")
val_df = load_split("val")


def make_labels(df):
    targets = np.zeros(
        (len(df), NUM_LABELS),
        dtype=np.float32,
    )

    for i, value in enumerate(df["labels"].astype(str)):
        indices = [
            int(x.strip())
            for x in value.split(",")
            if x.strip()
        ]
        targets[i, indices] = 1.0

    return targets


y_train = make_labels(train_df)
y_val = make_labels(val_df)

PREPROCESSING = {
    "repeat": {
        "normalize_repeat": True,
        "remove_symbols": False,
    },
    "minimal": {
        "normalize_repeat": False,
        "remove_symbols": False,
    },
    "repeat_symbols": {
        "normalize_repeat": True,
        "remove_symbols": True,
    },
}

TEXT_CACHE = {}


def get_texts(mode):
    if mode not in TEXT_CACHE:
        kwargs = PREPROCESSING[mode]

        def convert(df):
            return (
                df["text"]
                .fillna("")
                .astype(str)
                .apply(
                    lambda text: text_preprocessing.preprocess_text(
                        text, **kwargs
                    )
                )
                .tolist()
            )

        TEXT_CACHE[mode] = (
            convert(train_df),
            convert(val_df),
        )

    return TEXT_CACHE[mode]


print("Train:", train_df.shape, y_train.shape)
print("Validation:", val_df.shape, y_val.shape)

assert y_train.shape == (40000, NUM_LABELS)
assert y_val.shape == (5000, NUM_LABELS)

Train: (40000, 3) (40000, 44)
Validation: (5000, 3) (5000, 44)


In [ ]:
tokenizer = KoBERTTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
tokenizer.truncation_side ="right"

example_ids = tokenizer.encode("한국어 모델을 공유합니다.")
print("예제 토큰 ID:", example_ids)

assert example_ids == [
    2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3
], "KoBERT 토크나이저 설정을 확인해 주세요."

INPUT_CASE = {}

def encode_texts(texts, max_len):
  return tokenizer(
      texts,
      padding="max_length",
      truncation=True,
      max_length=max_len,
      return_attention_mask=True,
      return_token_type_ids=False,
      return_tensors="pt",
  )

def get_encoded_inputs(config):
  key = (
      config["preprocessing"],
      config["max_len"],
  )

  if key not in INPUT_CASE:
    train_texts, val_texts = get_texts(config["preprocessing"])

    INPUT_CASE[key] = (
        encode_texts(train_texts, config["max_len"]),
        encode_texts(val_texts, config["max_len"]),
    )

  return INPUT_CASE[key]

def input_statistics(texts, max_len):
  encoded = tokenizer(
      texts,
      add_special_tokens=True,
      truncation=False,
      padding=False,
      return_attention_mask=False,
      return_token_type_ids=False,
  )["input_ids"]

  lengths = np.array([len(ids) for ids in encoded])
  total_tokens = sum(len(ids) for ids in encoded)
  unknown_tokens = sum(ids.count(tokenizer.unk_token_id) for ids in encoded)

  return {
      "mean_tokens": float(lengths.mean()),
      "p95": float(np.percentile(lengths, 95)),
      "p99": float(np.percentile(lengths, 99)),
      "truncation_rate": float((lengths > max_len).mean()),
      "unk_ratio": unknown_tokens / max(total_tokens, 1),
  }

class KoBERTDataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = torch.as_tensor(labels, dtype=torch.float32)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, index):
    return {
        "input_ids": self.encodings["input_ids"][index],
        "attention_mask": self.encodings["attention_mask"][index],
        "labels": self.labels[index],
    }

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


예제 토큰 ID: [2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3]


In [ ]:
class KoBERTClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.bert = BertModel.from_pretrained(
            config["model_name"]
        )
        self.dropout = nn.Dropout(config["dropout"])
        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            NUM_LABELS,
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # 단일 문장이므로 token_type_ids는 기본값 0 사용
            return_dict=True,
        )

        features = self.dropout(outputs.pooler_output)
        return self.classifier(features)


BASE_CONFIG = {
    "implementation_version": 1,
    "model_name": MODEL_NAME,
    "seed": 42,
    "preprocessing": "repeat",
    "max_len": 128,
    "dropout": 0.1,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "batch_size": 16,
    "max_epochs": 5,
    "patience": 2,
    "warmup_ratio": 0.1,
    "loss": "bce",
}

train_texts, val_texts = get_texts("repeat")

initial_stats = {
    "train": input_statistics(train_texts, 128),
    "validation": input_statistics(val_texts, 128),
}

display(pd.DataFrame(initial_stats).T)
save_json(
    initial_stats,
    ARTIFACT_DIR / "initial_input_statistics.json",
)

,mean_tokens,p95,p99,truncation_rate,unk_ratio
train,37.075975,88.00,147.00,0.017275,0.008636
validation,37.891600,89.05,145.01,0.017600,0.008377


In [ ]:
THRESHOLDS = np.round(
    np.arange(0.10, 0.61, 0.05),
    2,
)


def classification_metrics(labels, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    return {
        "micro_f1": float(f1_score(
            labels, predictions,
            average="micro",
            zero_division=0,
        )),
        "macro_f1": float(f1_score(
            labels, predictions,
            average="macro",
            zero_division=0,
        )),
        "hamming_loss": float(hamming_loss(
            labels, predictions
        )),
        "precision_micro": float(precision_score(
            labels, predictions,
            average="micro",
            zero_division=0,
        )),
        "recall_micro": float(recall_score(
            labels, predictions,
            average="micro",
            zero_division=0,
        )),
        "predicted_labels_per_sample": float(
            predictions.sum(axis=1).mean()
        ),
    }


def threshold_search(
    labels,
    probabilities,
    thresholds=THRESHOLDS,
):
    table = pd.DataFrame([
        {
            "threshold": float(threshold),
            **classification_metrics(
                labels, probabilities, threshold
            ),
        }
        for threshold in thresholds
    ])

    best = table.loc[
        table["micro_f1"].idxmax()
    ].to_dict()

    return table, best


def labelwise_table(labels, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    return pd.DataFrame({
        "label_id": np.arange(NUM_LABELS),
        "label": LABEL_NAMES,
        "support": labels.sum(axis=0).astype(int),
        "predicted": predictions.sum(axis=0),
        "precision": precision_score(
            labels, predictions,
            average=None, zero_division=0,
        ),
        "recall": recall_score(
            labels, predictions,
            average=None, zero_division=0,
        ),
        "f1": f1_score(
            labels, predictions,
            average=None, zero_division=0,
        ),
    })


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    count = 0
    all_probabilities = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(
            DEVICE, non_blocking=True
        )
        attention_mask = batch["attention_mask"].to(
            DEVICE, non_blocking=True
        )
        labels = batch["labels"].to(
            DEVICE, non_blocking=True
        )

        with autocast_context():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        count += len(labels)

        all_probabilities.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / count,
        np.vstack(all_probabilities),
        np.vstack(all_labels),
    )

In [ ]:
set_seed(42)

train_encodings, _ = get_encoded_inputs(BASE_CONFIG)

smoke_dataset = KoBERTDataset(
    {
        key: value[:4]
        for key, value in train_encodings.items()
    },
    y_train[:4],
)

smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=4,
)

smoke_model = KoBERTClassifier(BASE_CONFIG).to(DEVICE)
smoke_model.train()

batch = next(iter(smoke_loader))

with autocast_context():
  logits = smoke_model(
      batch["input_ids"].to(DEVICE),
      batch["attention_mask"].to(DEVICE),
  )

  loss = nn.BCEWithLogitsLoss()(
      logits,
      batch["labels"].to(DEVICE),
  )

assert logits.shape == (4, NUM_LABELS)
assert torch.isfinite(loss)

In [ ]:
def run_experiment(config, force=False):
    config = copy.deepcopy(config)
    run_id = config_id(config)

    run_dir = ARTIFACT_DIR / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_path = run_dir / "summary.json"
    checkpoint_path = run_dir / "best.pt"

    if summary_path.exists() and not force:
        print("저장된 결과 사용:", run_id)
        return json.loads(
            summary_path.read_text(encoding="utf-8")
        )

    train_encodings, val_encodings = get_encoded_inputs(config)

    train_dataset = KoBERTDataset(
        train_encodings, y_train
    )
    val_dataset = KoBERTDataset(
        val_encodings, y_val
    )

    set_seed(config["seed"])

    generator = torch.Generator().manual_seed(
        config["seed"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    model = KoBERTClassifier(config).to(DEVICE)

    if config["loss"] == "weighted_bce":
        positives = y_train.sum(axis=0)
        negatives = len(y_train) - positives

        weights = np.minimum(
            5.0,
            np.sqrt(
                negatives / np.maximum(positives, 1)
            ),
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                weights,
                dtype=torch.float32,
                device=DEVICE,
            )
        )
    else:
        criterion = nn.BCEWithLogitsLoss()

    # bias와 LayerNorm 파라미터에는 weight decay 제외
    decay_params = []
    no_decay_params = []

    for name, parameter in model.named_parameters():
        if "bias" in name or "LayerNorm.weight" in name:
            no_decay_params.append(parameter)
        else:
            decay_params.append(parameter)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": decay_params,
                "weight_decay": config["weight_decay"],
            },
            {
                "params": no_decay_params,
                "weight_decay": 0.0,
            },
        ],
        lr=config["lr"],
    )

    total_steps = (
        len(train_loader) * config["max_epochs"]
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            total_steps * config["warmup_ratio"]
        ),
        num_training_steps=total_steps,
    )

    scaler = torch.amp.GradScaler("cuda")

    best_score = -1.0
    best_epoch = 0
    patience_counter = 0
    history = []

    save_json(config, run_dir / "config.json")

    synchronize()
    start = time.perf_counter()

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        total_loss = 0.0
        count = 0

        progress = tqdm(
            train_loader,
            desc=f"{run_id} | epoch {epoch}",
            leave=False,
        )

        for batch in progress:
            input_ids = batch["input_ids"].to(
                DEVICE, non_blocking=True
            )
            attention_mask = batch["attention_mask"].to(
                DEVICE, non_blocking=True
            )
            labels = batch["labels"].to(
                DEVICE, non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(
                    input_ids, attention_mask
                )
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Loss에 NaN 또는 Inf가 발생했습니다."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            # overflow로 optimizer step이 생략되면 scheduler도 생략
            if scaler.get_scale() >= old_scale:
                scheduler.step()

            total_loss += loss.item() * len(labels)
            count += len(labels)

            progress.set_postfix(
                loss=f"{total_loss / count:.4f}"
            )

        val_loss, probabilities, labels = evaluate(
            model, val_loader, criterion
        )
        _, best = threshold_search(
            labels, probabilities
        )

        record = {
            "epoch": epoch,
            "train_loss": total_loss / count,
            "val_loss": val_loss,
            **best,
        }
        history.append(record)

        print(
            f"{run_id} | epoch {epoch:02d} | "
            f"train {record['train_loss']:.4f} | "
            f"val {val_loss:.4f} | "
            f"micro {best['micro_f1']:.4f} | "
            f"macro {best['macro_f1']:.4f} | "
            f"threshold {best['threshold']:.2f}"
        )

        pd.DataFrame(history).to_csv(
            run_dir / "history.csv",
            index=False,
        )

        if best["micro_f1"] > best_score:
            best_score = best["micro_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    key: value.detach().cpu()
                    for key, value in model.state_dict().items()
                },
                checkpoint_path,
            )
        else:
            patience_counter += 1

            if patience_counter >= config["patience"]:
                print("조기 종료")
                break

    synchronize()
    elapsed = time.perf_counter() - start

    state = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(state)
    del state

    val_loss, probabilities, labels = evaluate(
        model, val_loader, criterion
    )
    threshold_table, best = threshold_search(
        labels, probabilities
    )

    threshold_table.to_csv(
        run_dir / "thresholds.csv", index=False
    )

    labelwise_table(
        labels, probabilities, best["threshold"]
    ).to_csv(
        run_dir / "labelwise.csv",
        index=False,
        encoding="utf-8-sig",
    )

    np.savez_compressed(
        run_dir / "validation_predictions.npz",
        probabilities=probabilities,
        labels=labels,
    )

    summary = {
        "run_id": run_id,
        "config": config,
        **best,
        "ap_micro": float(average_precision_score(
            labels, probabilities, average="micro"
        )),
        "ap_macro": float(average_precision_score(
            labels, probabilities, average="macro"
        )),
        "val_loss": float(val_loss),
        "best_epoch": best_epoch,
        "epochs_run": len(history),
        "training_seconds": elapsed,
        "parameters": sum(
            p.numel() for p in model.parameters()
        ),
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": str(torch.__version__),
        "transformers_version": transformers.__version__,
        "test_evaluated": False,
    }

    save_json(summary, summary_path)

    del model, optimizer, scheduler, scaler, criterion
    gc.collect()
    torch.cuda.empty_cache()

    return summary


def result_table(results):
    rows = []

    for result in results:
        rows.append({
            "run_id": result["run_id"],
            **result["config"],
            **{
                key: result[key]
                for key in [
                    "micro_f1",
                    "macro_f1",
                    "hamming_loss",
                    "ap_micro",
                    "ap_macro",
                    "threshold",
                    "best_epoch",
                    "epochs_run",
                    "training_seconds",
                ]
            },
        })

    return (
        pd.DataFrame(rows)
        .sort_values("micro_f1", ascending=False)
        .reset_index(drop=True)
    )


def compare_candidates(base, changes, stage):
    configs = [copy.deepcopy(base)]

    for change in changes:
        candidate = copy.deepcopy(base)
        candidate.update(change)
        configs.append(candidate)

    configs = {
        config_id(config): config
        for config in configs
    }

    results = [
        run_experiment(config)
        for config in configs.values()
    ]

    table = result_table(results)

    print("\nStage:", stage)
    display(table)

    table.to_csv(
        ARTIFACT_DIR / f"{stage}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    winner = max(
        results,
        key=lambda result: result["micro_f1"],
    )

    save_json(
        winner["config"],
        ARTIFACT_DIR / f"{stage}_winner.json",
    )

    return copy.deepcopy(winner["config"]), results

In [ ]:
baseline_result = run_experiment(BASE_CONFIG)
CURRENT = copy.deepcopy(BASE_CONFIG)

display(result_table([baseline_result]))

baseline_dir = (
    ARTIFACT_DIR / "runs" / baseline_result["run_id"]
)

history = pd.read_csv(
    baseline_dir / "history.csv"
)
display(history)

baseline_labels = pd.read_csv(
    baseline_dir / "labelwise.csv"
)

print("성능이 낮은 감정")
display(baseline_labels.sort_values("f1").head(10))

70f6cb83ba10 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

70f6cb83ba10 | epoch 01 | train 0.4168 | val 0.3380 | micro 0.5648 | macro 0.4147 | threshold 0.25


70f6cb83ba10 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

70f6cb83ba10 | epoch 02 | train 0.3262 | val 0.3203 | micro 0.5932 | macro 0.4564 | threshold 0.30


70f6cb83ba10 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

70f6cb83ba10 | epoch 03 | train 0.3012 | val 0.3103 | micro 0.6063 | macro 0.4815 | threshold 0.30


... (4 outputs left out)

,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,max_epochs,patience,warmup_ratio,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,70f6cb83ba10,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00002,0.01,16,5,2,0.1,bce,0.612402,0.49563,0.154218,0.660332,0.520715,0.3,5,5,1042.421449


,epoch,train_loss,val_loss,threshold,micro_f1,macro_f1,hamming_loss,precision_micro,recall_micro,predicted_labels_per_sample
0,1,0.416752,0.338040,0.25,0.564821,0.414729,0.184836,0.487944,0.670452,10.8164
1,2,0.326223,0.320259,0.30,0.593189,0.456427,0.160241,0.543417,0.652998,9.4594
2,3,0.301208,0.310278,0.30,0.606281,0.481494,0.154368,0.557562,0.664329,9.3794
3,4,0.285013,0.308434,0.35,0.611663,0.479748,0.145077,0.586892,0.638618,8.5658
4,5,0.274314,0.308155,0.30,0.612402,0.495630,0.154218,0.556379,0.680971,9.6348


성능이 낮은 감정


,label_id,label,support,predicted,precision,recall,f1
30,30,죄책감,89,0,0.000000,0.000000,0.000000
17,17,부끄러움,286,0,0.000000,0.000000,0.000000
25,25,패배/자기혐오,183,18,0.388889,0.038251,0.069652
26,26,귀찮음,312,40,0.375000,0.048077,0.085227
18,18,공포/무서움,134,14,0.571429,0.059701,0.108108
35,35,부담/안_내킴,578,263,0.311787,0.141869,0.195006
36,36,서러움,234,156,0.288462,0.192308,0.230769
19,19,절망,455,363,0.374656,0.298901,0.332518
27,27,힘듦/지침,477,434,0.405530,0.368973,0.386389
11,11,비장함,399,382,0.413613,0.395990,0.404609


In [ ]:
run_dir = ARTIFACT_DIR / "runs" / baseline_result["run_id"]

with np.load(run_dir / "validation_predictions.npz") as data:
    probabilities = data["probabilities"]
    labels = data["labels"]

rows = []

for label_id in [30, 17]:
    target = labels[:, label_id]
    scores = probabilities[:, label_id]

    positive = target == 1
    negative = ~positive

    grid = np.round(np.arange(0.01, 1.00, 0.01), 2)
    f1_values = [
        f1_score(
            target,
            scores >= threshold,
            zero_division=0,
        )
        for threshold in grid
    ]
    best_index = int(np.argmax(f1_values))

    rows.append({
        "label": LABEL_NAMES[label_id],
        "train_positive": int(y_train[:, label_id].sum()),
        "val_positive": int(target.sum()),
        "max_score": float(scores.max()),
        "positive_mean_score": float(scores[positive].mean()),
        "negative_mean_score": float(scores[negative].mean()),
        "AP": float(average_precision_score(target, scores)),
        "positive_rate": float(target.mean()),
        "best_label_threshold": float(grid[best_index]),
        "best_label_f1": float(f1_values[best_index]),
    })

display(pd.DataFrame(rows).T)

,0,1
label,죄책감,부끄러움
train_positive,698,2329
val_positive,89,286
max_score,0.27474,0.267408
positive_mean_score,0.061339,0.092548
negative_mean_score,0.01854,0.056779
AP,0.101262,0.124497
positive_rate,0.0178,0.0572
best_label_threshold,0.09,0.09
best_label_f1,0.206642,0.187838


In [ ]:
CURRENT, lr_results = compare_candidates(
    CURRENT,
    [
        {"lr": 1e-5},
        {"lr": 3e-5},
    ],
    stage="01_learning_rate",
)

저장된 결과 사용: 70f6cb83ba10


84689b7c81d4 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

84689b7c81d4 | epoch 01 | train 0.4414 | val 0.3498 | micro 0.5531 | macro 0.3753 | threshold 0.25


84689b7c81d4 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

84689b7c81d4 | epoch 02 | train 0.3378 | val 0.3288 | micro 0.5808 | macro 0.4321 | threshold 0.30


84689b7c81d4 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (12 outputs left out)

b504e86ab9a8 | epoch 04 | train 0.2780 | val 0.3084 | micro 0.6115 | macro 0.4979 | threshold 0.30


b504e86ab9a8 | epoch 5:   0%|          | 0/2500 [00:00<?, ?it/s]

b504e86ab9a8 | epoch 05 | train 0.2652 | val 0.3085 | micro 0.6118 | macro 0.4925 | threshold 0.35

Stage: 01_learning_rate


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,max_epochs,patience,warmup_ratio,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,70f6cb83ba10,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00002,0.01,16,5,2,0.1,bce,0.612402,0.495630,0.154218,0.660332,0.520715,0.30,5,5,1042.421449
1,b504e86ab9a8,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00003,0.01,16,5,2,0.1,bce,0.611842,0.492490,0.145205,0.662460,0.523408,0.35,5,5,1036.278552
2,84689b7c81d4,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00001,0.01,16,5,2,0.1,bce,0.603860,0.475135,0.156650,0.643239,0.496976,0.30,5,5,1037.704976


In [ ]:
CURRENT, epoch_results = compare_candidates(
    CURRENT,
    [
        {"max_epochs": 8},
    ],
    stage="02_epoch_budget",
)

CURRENT, dropout_results = compare_candidates(
    CURRENT,
    [
        {"dropout": 0.3},
    ],
    stage="03_dropout",
)

저장된 결과 사용: 70f6cb83ba10


c80ee975acfd | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

c80ee975acfd | epoch 01 | train 0.4329 | val 0.3429 | micro 0.5592 | macro 0.4084 | threshold 0.25


c80ee975acfd | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

c80ee975acfd | epoch 02 | train 0.3294 | val 0.3200 | micro 0.5952 | macro 0.4588 | threshold 0.30


c80ee975acfd | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (26 outputs left out)

294c71044536 | epoch 07 | train 0.2501 | val 0.3084 | micro 0.6161 | macro 0.5058 | threshold 0.35


294c71044536 | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

294c71044536 | epoch 08 | train 0.2436 | val 0.3091 | micro 0.6158 | macro 0.5057 | threshold 0.35
조기 종료

Stage: 03_dropout


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,max_epochs,patience,warmup_ratio,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,294c71044536,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,16,8,2,0.1,bce,0.617704,0.499346,0.140873,0.665911,0.530498,0.35,6,8,1660.437587
1,c80ee975acfd,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00002,0.01,16,8,2,0.1,bce,0.615049,0.513736,0.152205,0.663729,0.528609,0.30,7,8,1662.831893


In [ ]:
save_json(CURRENT, ARTIFACT_DIR / "current_before_exit.json")
print("현재 설정 저장 완료")

현재 설정 저장 완료


In [ ]:
import json

with open(
    ARTIFACT_DIR / "current_before_exit.json",
    encoding="utf-8",
) as f:
    CURRENT = json.load(f)

In [ ]:
CURRENT, batch_results = compare_candidates(
    CURRENT,
    [
        {"batch_size": 32},
    ],
    stage="04_batch_size",
)

save_json(
    CURRENT,
    ARTIFACT_DIR / "current_after_training.json",
)

저장된 결과 사용: 294c71044536


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

3c267bfe1115 | epoch 1:   0%|          | 0/1250 [00:00<?, ?it/s]

3c267bfe1115 | epoch 01 | train 0.4608 | val 0.3527 | micro 0.5466 | macro 0.3560 | threshold 0.25


3c267bfe1115 | epoch 2:   0%|          | 0/1250 [00:00<?, ?it/s]

... (10 outputs left out)

3c267bfe1115 | epoch 07 | train 0.2648 | val 0.3080 | micro 0.6137 | macro 0.5031 | threshold 0.30


3c267bfe1115 | epoch 8:   0%|          | 0/1250 [00:00<?, ?it/s]

3c267bfe1115 | epoch 08 | train 0.2597 | val 0.3082 | micro 0.6138 | macro 0.5036 | threshold 0.30

Stage: 04_batch_size


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,294c71044536,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,16,...,bce,0.617704,0.499346,0.140873,0.665911,0.530498,0.35,6,8,1660.437587
1,3c267bfe1115,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,32,...,bce,0.613802,0.503585,0.151886,0.661809,0.524597,0.30,8,8,519.462134


In [ ]:
CURRENT, preprocessing_results = compare_candidates(
    CURRENT,
    [
        {"preprocessing": "minimal"},
        {"preprocessing": "repeat_symbols"},
    ],
    stage="05_preprocessing",
)

# 실제 전처리 결과가 같은지도 확인
repeat_train, repeat_val = get_texts("repeat")
symbols_train, symbols_val = get_texts("repeat_symbols")

print(
    "repeat와 repeat_symbols가 다른 학습 문장 수:",
    sum(
        left != right
        for left, right in zip(repeat_train, symbols_train)
    ),
)
print(
    "repeat와 repeat_symbols가 다른 검증 문장 수:",
    sum(
        left != right
        for left, right in zip(repeat_val, symbols_val)
    ),
)

저장된 결과 사용: 294c71044536


6b5ecb00ec8a | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

6b5ecb00ec8a | epoch 01 | train 0.4364 | val 0.3425 | micro 0.5592 | macro 0.4040 | threshold 0.25


6b5ecb00ec8a | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

6b5ecb00ec8a | epoch 02 | train 0.3319 | val 0.3180 | micro 0.5964 | macro 0.4551 | threshold 0.30


6b5ecb00ec8a | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (25 outputs left out)

6b256c2027eb | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

6b256c2027eb | epoch 08 | train 0.2482 | val 0.3128 | micro 0.6111 | macro 0.4993 | threshold 0.35

Stage: 05_preprocessing


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,294c71044536,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,16,...,bce,0.617704,0.499346,0.140873,0.665911,0.530498,0.35,6,8,1660.437587
1,6b5ecb00ec8a,1,skt/kobert-base-v1,42,minimal,128,0.3,0.00002,0.01,16,...,bce,0.617537,0.513489,0.150623,0.665094,0.529663,0.30,7,8,1023.103936
2,6b256c2027eb,1,skt/kobert-base-v1,42,repeat_symbols,128,0.3,0.00002,0.01,16,...,bce,0.611888,0.499993,0.144836,0.659154,0.524427,0.35,7,8,1024.969080


repeat와 repeat_symbols가 다른 학습 문장 수: 25198
repeat와 repeat_symbols가 다른 검증 문장 수: 3180


In [ ]:
selected_train_texts, selected_val_texts = get_texts(
    CURRENT["preprocessing"]
)

selected_stats = {
    "train": input_statistics(
        selected_train_texts, CURRENT["max_len"]
    ),
    "validation": input_statistics(
        selected_val_texts, CURRENT["max_len"]
    ),
}

display(pd.DataFrame(selected_stats).T)

save_json(
    selected_stats,
    ARTIFACT_DIR / "selected_input_statistics.json",
)

if selected_stats["train"]["truncation_rate"] >= 0.01:
    CURRENT, length_results = compare_candidates(
        CURRENT,
        [{"max_len": 192}],
        stage="06_input_length",
    )
else:
    length_results = []
    print("잘리는 학습 문장이 1% 미만이므로 길이 비교 생략")

,mean_tokens,p95,p99,truncation_rate,unk_ratio
train,37.075975,88.00,147.00,0.017275,0.008636
validation,37.891600,89.05,145.01,0.017600,0.008377


저장된 결과 사용: 294c71044536


043c26b05abf | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

043c26b05abf | epoch 01 | train 0.4370 | val 0.3424 | micro 0.5615 | macro 0.3910 | threshold 0.25


043c26b05abf | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

043c26b05abf | epoch 02 | train 0.3323 | val 0.3169 | micro 0.5988 | macro 0.4579 | threshold 0.30


... (9 outputs left out)

043c26b05abf | epoch 07 | train 0.2495 | val 0.3066 | micro 0.6185 | macro 0.5113 | threshold 0.35


043c26b05abf | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

043c26b05abf | epoch 08 | train 0.2431 | val 0.3079 | micro 0.6177 | macro 0.5179 | threshold 0.30
조기 종료

Stage: 06_input_length


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,043c26b05abf,1,skt/kobert-base-v1,42,repeat,192,0.3,0.00002,0.01,16,...,bce,0.618727,0.513889,0.150459,0.668342,0.532500,0.30,6,8,1056.357007
1,294c71044536,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,16,...,bce,0.617704,0.499346,0.140873,0.665911,0.530498,0.35,6,8,1660.437587


In [ ]:
baseline_result = run_experiment(BASE_CONFIG)

저장된 결과 사용: 70f6cb83ba10


In [ ]:
CURRENT, loss_results = compare_candidates(
    CURRENT,
    [
        {"loss": "weighted_bce"},
    ],
    stage="07_loss",
)

save_json(
    CURRENT,
    ARTIFACT_DIR / "final_config.json",
)

final_result = run_experiment(CURRENT)

print("최종 선택 설정")
display(pd.Series(CURRENT))
display(result_table([baseline_result, final_result]))

저장된 결과 사용: 043c26b05abf
저장된 결과 사용: 20e8cfc562ce

Stage: 07_loss


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,043c26b05abf,1,skt/kobert-base-v1,42,repeat,192,0.3,0.00002,0.01,16,...,bce,0.618727,0.513889,0.150459,0.668342,0.532500,0.3,6,8,1056.357007
1,20e8cfc562ce,1,skt/kobert-base-v1,42,repeat,192,0.3,0.00002,0.01,16,...,weighted_bce,0.614202,0.533135,0.148250,0.657027,0.539155,0.5,6,8,1062.336202


저장된 결과 사용: 043c26b05abf
최종 선택 설정


,0
implementation_version,1
model_name,skt/kobert-base-v1
seed,42
preprocessing,repeat
max_len,192
dropout,0.3
lr,0.00002
weight_decay,0.01
batch_size,16
max_epochs,8


,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,043c26b05abf,1,skt/kobert-base-v1,42,repeat,192,0.3,0.00002,0.01,16,...,bce,0.618727,0.513889,0.150459,0.668342,0.532500,0.3,6,8,1056.357007
1,70f6cb83ba10,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00002,0.01,16,...,bce,0.612402,0.495630,0.154218,0.660332,0.520715,0.3,5,5,1042.421449


In [ ]:
def refine_thresholds(result):
    run_dir = (
        ARTIFACT_DIR / "runs" / result["run_id"]
    )

    with np.load(
        run_dir / "validation_predictions.npz"
    ) as data:
        probabilities = data["probabilities"]
        labels = data["labels"]

    fine_grid = np.round(
        np.arange(0.01, 1.00, 0.01), 2
    )

    table, best = threshold_search(
        labels, probabilities, fine_grid
    )

    table.to_csv(
        run_dir / "fine_thresholds.csv",
        index=False,
    )

    global_summary = {
        **best,
        "ap_micro": result["ap_micro"],
        "ap_macro": result["ap_macro"],
    }

    save_json(
        global_summary,
        run_dir / "fine_global_summary.json",
    )

    labelwise_thresholds = np.full(
        NUM_LABELS,
        best["threshold"],
        dtype=np.float32,
    )

    for label_id in range(NUM_LABELS):
        # 검증셋에 양성 사례가 없는 감정은 공통 임계값 유지
        if labels[:, label_id].sum() == 0:
            continue

        scores = [
            f1_score(
                labels[:, label_id],
                probabilities[:, label_id] >= threshold,
                zero_division=0,
            )
            for threshold in fine_grid
        ]

        labelwise_thresholds[label_id] = fine_grid[
            int(np.argmax(scores))
        ]

    labelwise_metrics = classification_metrics(
        labels, probabilities, labelwise_thresholds
    )

    pd.DataFrame({
        "label": LABEL_NAMES,
        "threshold": labelwise_thresholds,
    }).to_csv(
        run_dir / "labelwise_thresholds.csv",
        index=False,
        encoding="utf-8-sig",
    )

    comparison = pd.DataFrame([
        {
            "method": "coarse_global",
            **classification_metrics(
                labels,
                probabilities,
                result["threshold"],
            ),
        },
        {
            "method": "fine_global",
            **classification_metrics(
                labels,
                probabilities,
                best["threshold"],
            ),
        },
        {
            "method": "labelwise",
            **labelwise_metrics,
        },
    ])

    comparison.to_csv(
        run_dir / "threshold_method_comparison.csv",
        index=False,
    )

    return global_summary, comparison


fine_summary, threshold_comparison = refine_thresholds(
    final_result
)

print("세밀한 공통 임계값:", fine_summary["threshold"])
display(threshold_comparison)

세밀한 공통 임계값: 0.31


,method,micro_f1,macro_f1,hamming_loss,precision_micro,recall_micro,predicted_labels_per_sample
0,coarse_global,0.618727,0.513889,0.150459,0.565944,0.682368,9.4914
1,fine_global,0.619191,0.512315,0.148391,0.572399,0.674314,9.2736
2,labelwise,0.607528,0.536829,0.162273,0.535464,0.702007,10.3204


In [ ]:
repeat_rows = []

for setting, template in [
    ("baseline", BASE_CONFIG),
    ("final", CURRENT),
]:
    for seed in [42, 43, 44]:
        config = copy.deepcopy(template)
        config["seed"] = seed

        result = run_experiment(config)
        fine, _ = refine_thresholds(result)

        repeat_rows.append({
            "setting": setting,
            "seed": seed,
            "run_id": result["run_id"],
            "micro_f1": fine["micro_f1"],
            "macro_f1": fine["macro_f1"],
            "hamming_loss": fine["hamming_loss"],
            "ap_micro": result["ap_micro"],
            "ap_macro": result["ap_macro"],
            "threshold": fine["threshold"],
            "training_seconds": result["training_seconds"],
            "gpu": result["gpu"],
        })

repeat_table = pd.DataFrame(repeat_rows)

repeat_summary = (
    repeat_table.groupby("setting")[
        [
            "micro_f1",
            "macro_f1",
            "hamming_loss",
            "ap_micro",
            "ap_macro",
            "training_seconds",
        ]
    ]
    .agg(["mean", "std"])
)

display(repeat_table)
display(repeat_summary)

repeat_table.to_csv(
    ARTIFACT_DIR / "repeat_results.csv",
    index=False,
    encoding="utf-8-sig",
)

repeat_summary.to_csv(
    ARTIFACT_DIR / "repeat_summary.csv",
    encoding="utf-8-sig",
)

저장된 결과 사용: 70f6cb83ba10


690906d7a1bf | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

690906d7a1bf | epoch 01 | train 0.4116 | val 0.3320 | micro 0.5775 | macro 0.4167 | threshold 0.30


690906d7a1bf | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

690906d7a1bf | epoch 02 | train 0.3181 | val 0.3107 | micro 0.6087 | macro 0.4788 | threshold 0.30


690906d7a1bf | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (43 outputs left out)

a467df13c675 | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

a467df13c675 | epoch 08 | train 0.2400 | val 0.3065 | micro 0.6213 | macro 0.5208 | threshold 0.30
조기 종료


,setting,seed,run_id,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,training_seconds,gpu
0,baseline,42,70f6cb83ba10,0.612762,0.493119,0.152014,0.660332,0.520715,0.31,1042.421449,NVIDIA L4
1,baseline,43,690906d7a1bf,0.620560,0.506641,0.146686,0.670958,0.535121,0.31,644.275258,NVIDIA A100-SXM4-80GB
2,baseline,44,1dbc5dff6a78,0.622816,0.507740,0.143045,0.672946,0.537468,0.33,644.021006,NVIDIA A100-SXM4-80GB
3,final,42,043c26b05abf,0.619191,0.512315,0.148391,0.668342,0.532500,0.31,1056.357007,NVIDIA A100-SXM4-80GB
4,final,43,4e1644b124a1,0.622551,0.516681,0.146305,0.673850,0.541851,0.30,930.476093,NVIDIA A100-SXM4-80GB
5,final,44,a467df13c675,0.623056,0.516749,0.146727,0.672894,0.539370,0.32,1060.998371,NVIDIA A100-SXM4-80GB


micro_f1            macro_f1           hamming_loss            \
              mean       std      mean       std         mean       std   
setting                                                                   
baseline  0.618713  0.005275  0.502500  0.008143     0.147248  0.004510   
final     0.621599  0.002101  0.515248  0.002540     0.147141  0.001103   

          ap_micro            ap_macro           training_seconds              
              mean       std      mean       std             mean         std  
setting                                                                        
baseline  0.668079  0.006782  0.531101  0.009071       776.905904  229.943242  
final     0.671695  0.002943  0.537907  0.004844      1015.943824   74.053598

In [ ]:
all_results = []

for path in sorted(
    (ARTIFACT_DIR / "runs").glob("*/summary.json")
):
    all_results.append(
        json.loads(path.read_text(encoding="utf-8"))
    )

all_experiments = result_table(all_results)

display(all_experiments)

all_experiments.to_csv(
    ARTIFACT_DIR / "all_experiments.csv",
    index=False,
    encoding="utf-8-sig",
)


def load_label_results(result):
    run_dir = (
        ARTIFACT_DIR / "runs" / result["run_id"]
    )

    with np.load(
        run_dir / "validation_predictions.npz"
    ) as data:
        probabilities = data["probabilities"]
        labels = data["labels"]

    fine = json.loads(
        (run_dir / "fine_global_summary.json").read_text(
            encoding="utf-8"
        )
    )

    return labelwise_table(
        labels, probabilities, fine["threshold"]
    )


baseline_label_table = load_label_results(
    baseline_result
)
final_label_table = load_label_results(
    final_result
)

label_comparison = final_label_table.merge(
    baseline_label_table[["label_id", "f1"]].rename(
        columns={"f1": "baseline_f1"}
    ),
    on="label_id",
)

label_comparison["f1_change"] = (
    label_comparison["f1"]
    - label_comparison["baseline_f1"]
)

print("최종 성능이 낮은 감정")
display(
    label_comparison.sort_values("f1").head(10)
)

print("개선 폭이 큰 감정")
display(
    label_comparison.sort_values(
        "f1_change", ascending=False
    ).head(10)
)

label_comparison.to_csv(
    ARTIFACT_DIR / "label_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("최종 모델 checkpoint:")
print(
    ARTIFACT_DIR / "runs"
    / final_result["run_id"] / "best.pt"
)

,run_id,implementation_version,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,batch_size,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,a467df13c675,1,skt/kobert-base-v1,44,repeat,192,0.3,0.00002,0.01,16,...,bce,0.622927,0.510790,0.141773,0.672894,0.539370,0.35,6,8,1060.998371
1,4e1644b124a1,1,skt/kobert-base-v1,43,repeat,192,0.3,0.00002,0.01,16,...,bce,0.622551,0.516681,0.146305,0.673850,0.541851,0.30,5,7,930.476093
2,1dbc5dff6a78,1,skt/kobert-base-v1,44,repeat,128,0.1,0.00002,0.01,16,...,bce,0.621980,0.512826,0.149191,0.672946,0.537468,0.30,5,5,644.021006
3,690906d7a1bf,1,skt/kobert-base-v1,43,repeat,128,0.1,0.00002,0.01,16,...,bce,0.620435,0.509037,0.148727,0.670958,0.535121,0.30,5,5,644.275258
4,043c26b05abf,1,skt/kobert-base-v1,42,repeat,192,0.3,0.00002,0.01,16,...,bce,0.618727,0.513889,0.150459,0.668342,0.532500,0.30,6,8,1056.357007
5,294c71044536,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,16,...,bce,0.617704,0.499346,0.140873,0.665911,0.530498,0.35,6,8,1660.437587
6,6b5ecb00ec8a,1,skt/kobert-base-v1,42,minimal,128,0.3,0.00002,0.01,16,...,bce,0.617537,0.513489,0.150623,0.665094,0.529663,0.30,7,8,1023.103936
7,c80ee975acfd,1,skt/kobert-base-v1,42,repeat,128,0.1,0.00002,0.01,16,...,bce,0.615049,0.513736,0.152205,0.663729,0.528609,0.30,7,8,1662.831893
8,20e8cfc562ce,1,skt/kobert-base-v1,42,repeat,192,0.3,0.00002,0.01,16,...,weighted_bce,0.614202,0.533135,0.148250,0.657027,0.539155,0.50,6,8,1062.336202
9,3c267bfe1115,1,skt/kobert-base-v1,42,repeat,128,0.3,0.00002,0.01,32,...,bce,0.613802,0.503585,0.151886,0.661809,0.524597,0.30,8,8,519.462134


최종 성능이 낮은 감정


,label_id,label,support,predicted,precision,recall,f1,baseline_f1,f1_change
17,17,부끄러움,286,7,0.428571,0.010490,0.020478,0.000000,0.020478
30,30,죄책감,89,1,1.000000,0.011236,0.022222,0.000000,0.022222
26,26,귀찮음,312,103,0.330097,0.108974,0.163855,0.080925,0.082931
25,25,패배/자기혐오,183,59,0.406780,0.131148,0.198347,0.060606,0.137741
35,35,부담/안_내킴,578,305,0.334426,0.176471,0.231031,0.181595,0.049435
36,36,서러움,234,185,0.308108,0.243590,0.272076,0.219839,0.052237
18,18,공포/무서움,134,73,0.397260,0.216418,0.280193,0.068966,0.211228
19,19,절망,455,379,0.374670,0.312088,0.340528,0.328767,0.011760
27,27,힘듦/지침,477,397,0.440806,0.366876,0.400458,0.385218,0.015239
9,9,우쭐댐/무시함,772,675,0.438519,0.383420,0.409122,0.402787,0.006335


개선 폭이 큰 감정


,label_id,label,support,predicted,precision,recall,f1,baseline_f1,f1_change
18,18,공포/무서움,134,73,0.397260,0.216418,0.280193,0.068966,0.211228
25,25,패배/자기혐오,183,59,0.406780,0.131148,0.198347,0.060606,0.137741
26,26,귀찮음,312,103,0.330097,0.108974,0.163855,0.080925,0.082931
36,36,서러움,234,185,0.308108,0.243590,0.272076,0.219839,0.052237
35,35,부담/안_내킴,578,305,0.334426,0.176471,0.231031,0.181595,0.049435
32,32,흐뭇함(귀여움/예쁨),518,556,0.541367,0.581081,0.560521,0.515209,0.045312
38,38,불쌍함/연민,668,795,0.457862,0.544910,0.497608,0.463364,0.034244
37,37,재미없음,467,343,0.641399,0.471092,0.543210,0.517588,0.025622
5,5,슬픔,502,643,0.471229,0.603586,0.529258,0.506329,0.022929
30,30,죄책감,89,1,1.000000,0.011236,0.022222,0.000000,0.022222


최종 모델 checkpoint:
/content/drive/MyDrive/26_KOTE/artifacts/week9_kobert/runs/043c26b05abf/best.pt


In [ ]:
kobert_means = repeat_table.groupby(
    "setting"
)["micro_f1"].mean()

model_comparison = pd.DataFrame([
    {
        "model": "TextCNN — Week 8 final",
        "validation_micro_f1_mean": 0.5691,
    },
    {
        "model": "BiLSTM — Week 8 final",
        "validation_micro_f1_mean": 0.5846,
    },
    {
        "model": "KoBERT — initial",
        "validation_micro_f1_mean": kobert_means["baseline"],
    },
    {
        "model": "KoBERT — final",
        "validation_micro_f1_mean": kobert_means["final"],
    },
])

display(model_comparison)

model_comparison.to_csv(
    ARTIFACT_DIR / "model_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

,model,validation_micro_f1_mean
0,TextCNN — Week 8 final,0.569100
1,BiLSTM — Week 8 final,0.584600
2,KoBERT — initial,0.618713
3,KoBERT — final,0.621599
